# APS Failure Prediction - v1 benchmark (reference only)

**This notebook is kept for reference and is not part of the service.** It is the
original exploratory work: clean the heavy-truck APS sensor data, then benchmark
Random Forest against XGBoost.

It is excluded from ruff, mypy, the test suite and the Docker image. The
`service` version of this project reimplements every step below as a tested
module - see `app/ml/` - and fixes two things this notebook got wrong:

1. **It evaluates a classification problem with regression metrics.**
   R², RMSE and MAE are computed on a 0/1 target below. R² on a binary outcome
   is not interpretable, and on a dataset that is ~1.7% positive a model that
   predicts "never fails" for every truck scores a *great* MAE while being
   worthless. The service reports ROC-AUC, PR-AUC, precision, recall and the
   official Scania cost metric (10xFP + 500xFN) instead, and still prints these
   three under `legacy_regression` so the numbers remain comparable.

2. **The cleaning decisions live in notebook cells, so inference cannot
   reproduce them.** Which columns were dropped for missingness, and what the
   imputation medians were, exist only in this kernel's memory. The service
   turns these into `DataCleaner` and `FeaturePipeline` fit/transform classes
   that are serialised with the model, so a prediction at serving time sees
   exactly the transformations training saw.

Other things the service added: a decision threshold tuned for cost (the
default 0.5 throws away most of the model's value at a 50:1 cost ratio), model
versioning with metadata, and drift detection against the recorded training
distribution.


## 1. Setup

The real dataset is **APS Failure at Scania Trucks** (UCI / Scania): ~60k rows,
171 anonymised sensor columns, `class` of `pos`/`neg`, missing values encoded
as the literal string `na`.

The repo ships a *synthetic* stand-in with the same shape and pathologies, so
this notebook runs without the real download. Point `DATA_PATH` at the real CSV
for real numbers.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 50)

DATA_PATH = "../data/sample/aps_sample_train.csv"
RANDOM_STATE = 42

In [ ]:
df = pd.read_csv(DATA_PATH, dtype=str, low_memory=False)
print(df.shape)
df.head()

## 2. Explore the target and missingness

Two facts dominate this dataset: the class imbalance, and how much is missing.

In [ ]:
df["class"].value_counts(normalize=True)

In [ ]:
# `na` is a string, so every sensor column reads as object until coerced.
features = df.drop(columns=["class"]).replace("na", np.nan).apply(pd.to_numeric, errors="coerce")
missing = features.isna().mean().sort_values(ascending=False)
missing.head(10)

In [ ]:
missing.describe()

## 3. Clean

Drop columns that are mostly missing, drop constant columns, median-impute the
rest.

**This is the step the service had to extract.** The `DROP_THRESHOLD`, the exact
list of dropped columns and the imputer's learned medians all live in this
kernel - so a model trained here cannot be served without guessing at them.

In [ ]:
DROP_THRESHOLD = 0.7

y = (df["class"].str.strip().str.lower() == "pos").astype(int)

high_missing = missing[missing > DROP_THRESHOLD].index.tolist()
X = features.drop(columns=high_missing)
print(f"dropped {len(high_missing)} mostly-missing columns: {high_missing}")

constant = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
X = X.drop(columns=constant)
print(f"dropped {len(constant)} constant columns: {constant}")

imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
print(X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())

## 4. Benchmark Random Forest vs XGBoost

Both get class-imbalance handling: `class_weight` for the forest,
`scale_pos_weight` for XGBoost.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf.fit(X_train, y_train)
rf_pred = rf.predict_proba(X_test)[:, 1]

In [ ]:
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict_proba(X_test)[:, 1]

## 5. Score them

### ⚠️ This is the methodological error

R², RMSE and MAE are **regression** metrics and this is a binary classification
problem. They are computed here because that is what v1 did; the numbers are
not wrong arithmetically, they are the wrong *question*.

The cell after this one shows why: a model that never predicts a failure - the
most useless possible model for this problem - wins on MAE.

In [ ]:
def regression_report(name, y_true, y_score):
    return {
        "model": name,
        "r2": r2_score(y_true, y_score),
        "rmse": root_mean_squared_error(y_true, y_score),
        "mae": mean_absolute_error(y_true, y_score),
    }

pd.DataFrame([
    regression_report("random_forest", y_test, rf_pred),
    regression_report("xgboost", y_test, xgb_pred),
]).set_index("model")

In [ ]:
# The "never predicts a failure" baseline, scored the same way.
# Its MAE beats both real models, and it would miss every single breakdown.
never = np.zeros_like(y_test, dtype=float)
pd.DataFrame([
    regression_report("random_forest", y_test, rf_pred),
    regression_report("xgboost", y_test, xgb_pred),
    regression_report("never_predicts_failure", y_test, never),
]).set_index("model").sort_values("mae")

## 6. What the service does instead

`app/ml/evaluation.py` reports:

* **ROC-AUC / PR-AUC** - ranking quality, with PR-AUC the one that matters
  under heavy imbalance
* **precision / recall / F1 / confusion matrix** at an explicit threshold
* **the official Scania cost**, `10 x FP + 500 x FN`, which is what model
  selection actually uses
* the three metrics above, namespaced under `legacy_regression`, so this
  notebook's numbers stay comparable

It also **tunes the decision threshold to minimise cost**. At a 50:1
false-negative penalty the cost-optimal threshold is far below 0.5, so leaving
it at the default discards most of the model's value.

Run the real thing with:

```bash
python scripts/train.py            # synthetic sample
python scripts/train.py --data path/to/aps_failure_training_set.csv
```